<a href="https://colab.research.google.com/github/Yaminipampana/CODSOFT/blob/main/Movie_rating_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import io
imdb_df = pd.read_csv((io.BytesIO(uploaded['IMDb Movies India.csv'])),encoding='unicode_escape')

In [ ]:
imdb_df.head(10)

In [ ]:
imdb_df.shape

***Data*** ***Cleaning***

In [ ]:
imdb_df.isnull().sum()

In [ ]:
imdb_df.info()

In [ ]:
imdb_df.duplicated().sum()

In [ ]:
imdb_df.dropna(inplace=True)

In [ ]:
imdb_df.shape

In [ ]:
imdb_df.isnull().sum()

In [ ]:
imdb_df.drop_duplicates(inplace=True)

In [ ]:
imdb_df.shape

In [ ]:
imdb_df.columns

***Data*** ***Pre***-***Processing***

In [ ]:
# Inspect the 'Year' column
print(imdb_df['Year'].unique())

# Drop rows with missing values in the 'Year' column
imdb_df.dropna(subset=['Year'], inplace=True)

# Convert 'Year' column to integer
# The 'Year' column is already in integer format, so this conversion is not needed.
# imdb_df['Year'] = imdb_df['Year'].str.replace(r'[()]', '', regex=True).astype(int)

In [ ]:
# Convert 'Duration' column to numeric (removing ' min' and converting to integers)
# The 'Duration' column is already in integer format, so this conversion is not needed.
# imdb_df['Duration'] = pd.to_numeric(imdb_df['Duration'].str.replace('min', ''))

In [ ]:
imdb_df['Genre'] = imdb_df['Genre'].str.split(', ')
imdb_df = imdb_df.explode('Genre')
imdb_df['Genre'].fillna(imdb_df['Genre'].mode()[0], inplace=True)

In [ ]:
imdb_df['Votes'] = imdb_df['Votes'].astype(str).str.replace(',', '', regex=False)
imdb_df['Votes'] = imdb_df['Votes'].replace('nan', np.nan) # Replace 'nan' strings with actual NaN
imdb_df['Votes'] = pd.to_numeric(imdb_df['Votes'], errors='coerce') # Use errors='coerce' to handle other parsing issues

In [ ]:
imdb_df.info()

***Data*** ***Visualizing***

In [ ]:
year = px.histogram(imdb_df, x = 'Year', histnorm='probability density', nbins= 30)
year.show()

In [ ]:
# Group data by Year and calculate the average rating
avg_rating_by_year = imdb_df.groupby(['Year', 'Genre'])['Rating'].mean().reset_index()

#Get the top 10 genres
top_genres = imdb_df['Genre'].value_counts().head().index

# Filter the data to include only the top 3 genres
average_rating_by_year = avg_rating_by_year[avg_rating_by_year['Genre'].isin(top_genres)]

#Create the line plot with Plotly Express
fig = px.line(avg_rating_by_year, x='Year', y='Rating', color = "Genre")

# Updating the detals into chart like title and hue
fig.update_layout(title="Average Rating by Year for Top Genres', xaxis title='Year', yaxis_title='Average Rating")

#Show the plot
fig.show()

In [ ]:
rating_fig = px.histogram(imdb_df, x = 'Rating', histnorm='probability density', nbins=40)
rating_fig.update_layout(title='Distribution of Rating', title_x=0.5, title_pad=dict(t=20), title_font=dict(size=20), xaxis_title='Rating', yaxis_title='Probability density')
rating_fig.show()

***Feature*** ***Engineering***

In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# imdb_df.drop('Name', axis = 1, inplace = True)

In [ ]:
genre_mean_rating = imdb_df.groupby('Genre') ['Rating'].transform('mean')
imdb_df['Genre_mean_rating'] = genre_mean_rating

director_mean_rating = imdb_df.groupby('Director') ['Rating'].transform('mean')
imdb_df['Director_encoded'] = director_mean_rating

actor1_mean_rating = imdb_df.groupby('Actor 1') ['Rating'].transform('mean')
imdb_df['Actor1_encoded'] = actor1_mean_rating

actor2_mean_rating = imdb_df.groupby('Actor 2') ['Rating'].transform('mean')
imdb_df['Actor2_encoded'] = actor2_mean_rating

actor3_mean_rating = imdb_df.groupby('Actor 3') ['Rating'].transform('mean')
imdb_df['Actor3_encoded'] = actor3_mean_rating

In [ ]:
X = imdb_df[['Year','Votes','Duration','Genre_mean_rating','Director_encoded','Actor1_encoded','Actor2_encoded','Actor3_encoded']]
y = imdb_df['Rating']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2,random_state=42)

***Model*** ***Buliding***

In [ ]:
Model = LinearRegression()
Model.fit(X_train, y_train)
Model_pred = Model.predict(X_test)

In [ ]:
print('The performace evalution of Logistic Regession is below: ','\n')
print('Mean squared error: ',mean_squared_error(y_test, Model_pred))
print('Mean absolute error: ',mean_absolute_error(y_test, Model_pred))
print('R2 score: ',r2_score(y_test, Model_pred))

***Model*** ***Testing***

In [ ]:
X.head(5)

In [ ]:
y.head(5)

In [ ]:
data = {'Year':[2019], 'Votes':[36],'Duration':[111],'Genre_mean_rating':[5.8],'Director_encoded':[4.5],'Actor1_encoded':[5.3],'Actor2_encoded':[4.5], 'Actor3_encoded':[5.0]}
trail = pd.DataFrame(data)

In [ ]:
rating_predicted = Model.predict(trail)
print("Predicted Rating: ", rating_predicted[0])